# Mart: IP charges (annual time-grain) fact table

Reads the staged OECD IP charges data (long — one row per accounting entry) and
writes the final dashboard-ready **wide** fact table (one row per year × counterpart).

| | |
|---|---|
| **Reads** | `data/staging/oecd/stg_oecd__ip_charges.csv` |
| **Writes** | `data/marts/fct_ip_charges_yearly.csv` |

**Output schema** (columns follow the OECD Data Explorer's general-audience labels):

| Column | Type | Description |
|---|---|---|
| `year` | int | Reference year |
| `counterpart_country` | str | Trading partner full name (e.g. `"United States"`) |
| `balance_cad_millions` | float | Net = revenue − expenditure, millions CAD (negative = Canada paid more) |
| `revenue_cad_millions` | float | Revenue: royalties Canada **earns**, millions CAD (null for some partner-years) |
| `expenditure_cad_millions` | float | Expenditure: royalties Canada **pays**, millions CAD (null for some partner-years) |

In [ ]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import STG_OECD_DIR, MARTS_DIR
from utils.oecd_utils import pivot_ip_charges_wide

MARTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
INPUT_FILENAME  = 'stg_oecd__ip_charges.csv'
OUTPUT_FILENAME = 'fct_ip_charges_yearly.csv'

df = pd.read_csv(STG_OECD_DIR / INPUT_FILENAME)
print(f'Loaded {len(df)} rows from staging (long: one row per accounting entry)')
df.head()

In [ ]:
# 1) Pivot the long staged data (one row per accounting entry) into the wide fact
#    shape: one row per (year, counterpart) with balance/revenue/expenditure columns.
wide = pivot_ip_charges_wide(df)

# 2) Align OECD country names to the Dim MacroRegion naming convention in Power BI.
#    Mapping uses ISO area codes (counterpart_area) to avoid special-character issues.
AREA_CODE_NAME_MAP = {
    'CHN': 'China',
    'HKG': 'Hong Kong',
    'SVK': 'Slovakia',
    'KOR': 'South Korea',
    'TWN': 'Taiwan',
}

fct_df = (
    wide
    .assign(counterpart_country=lambda x: x['counterpart_area'].map(AREA_CODE_NAME_MAP).fillna(x['counterpart_country']))
    [['year', 'counterpart_country',
      'balance_cad_millions', 'revenue_cad_millions', 'expenditure_cad_millions']]
    .sort_values(['year', 'counterpart_country'])
    .reset_index(drop=True)
)

print(f'Final mart shape: {fct_df.shape}')
print(f'Countries: {sorted(fct_df["counterpart_country"].unique().tolist())}')
fct_df.head(10)

In [ ]:
output_path = MARTS_DIR / OUTPUT_FILENAME
fct_df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Fact table saved to: {output_path}')